# Empirical evaluations (Task 2)

# Further evaluations

...

# Evaluation of the resource tasks (1.6–1.8)

Evaluation of the combined resource behaviour:

1. **Permission coverage:** every activity has at least one permitted resource (no deadlock), both modes.
2. **Availability faithfulness:** share of real events that fall inside each availability model.
3. **Permission breadth:** basic resource–activity matrix vs. advanced role-based model.

The patched dev-core ("SimulationEngineCore.py", handoff fixes) compiles; a full simulated-vs-real run (utilization, waiting/cycle times) is added here once the integrated engine is available.

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
from resources.log_loader import load_slim_log
from resources.permissions import PermissionModel
from resources.availability import AvailabilityModel, learn_calendars

df = load_slim_log('../data/BPI Challenge 2017.xes', '../data/bpic17_slim.parquet')
ts = pd.to_datetime(df['time:timestamp'], utc=True)
df = df.assign(weekday=ts.dt.weekday, hour=ts.dt.hour)
activities = sorted(df['concept:name'].dropna().astype(str).unique())

## 1. Permission coverage (no deadlock)

In [3]:
basic_p = PermissionModel(df, artifact_path=None)
adv_p = PermissionModel(df, artifact_path='../results/permissions_roles.json')
cov_basic = sum(len(basic_p.who_can(a)) > 0 for a in activities)
cov_adv = sum(len(adv_p.who_can(a)) > 0 for a in activities)
print('permission mode (advanced):', adv_p.mode)
print('activities with >=1 permitted resource | basic: %d/%d | advanced: %d/%d' % (
    cov_basic, len(activities), cov_adv, len(activities)))

permission mode (advanced): advanced
activities with >=1 permitted resource | basic: 26/26 | advanced: 26/26


## 2. Availability faithfulness (event coverage vs real log)

In [4]:
cal = learn_calendars(df)
cal_t = {r: {(int(w), int(h)) for w, h in b} for r, b in cal.items()}
in_basic = ((df['weekday'] < 5) & (df['hour'] >= 8) & (df['hour'] < 18)).mean()
samp = df.sample(min(100000, len(df)), random_state=1)
in_adv = np.mean([(int(w), int(h)) in cal_t.get(str(r), set())
                  for r, w, h in zip(samp['org:resource'], samp['weekday'], samp['hour'])])
print('availability event-coverage | basic interval: %.1f%% | learned calendars: %.1f%%' % (
    100*in_basic, 100*in_adv))

availability event-coverage | basic interval: 72.0% | learned calendars: 96.4%


## 3. Permission breadth (basic vs advanced) + summary

In [5]:
added = sum(len(adv_p.who_can(a) - basic_p.who_can(a)) for a in activities)
summary = pd.DataFrame([
    ['1.6 availability', 'event coverage of real log', '%.1f%%' % (100*in_basic), '%.1f%%' % (100*in_adv)],
    ['1.7 permissions', 'activities covered (no deadlock)', f'{cov_basic}/{len(activities)}', f'{cov_adv}/{len(activities)}'],
    ['1.7 permissions', 'extra role-based (act,res) permissions', '0', str(added)],
], columns=['task', 'metric', 'basic', 'advanced'])
summary.to_csv('../results/resource_eval_summary.csv', index=False)
print(summary.to_string(index=False))
print('\nsaved ../results/resource_eval_summary.csv')

            task                                 metric basic advanced
1.6 availability             event coverage of real log 72.0%    96.4%
 1.7 permissions       activities covered (no deadlock) 26/26    26/26
 1.7 permissions extra role-based (act,res) permissions     0      872

saved ../results/resource_eval_summary.csv


**Summary:** Both advanced models clearly improve on their basic counterparts on real data: per-resource calendars track ~96% of real events vs ~72% for the fixed interval, and OrdinoR role discovery adds hundreds of role-based permissions while keeping every activity covered. The next step is the integrated simulated-vs-real evaluation (utilization, waiting/cycle times), which requires K1's real XES loading and the K2/K3 engines.